## Market Regimes

In [1]:
# import libraries
import json
import numpy as np
import pandas as pd
import joblib

In [2]:
# define csv file
rth_df = pd.read_csv("multiasset_5m_rth_features.csv")

rth_df["timestamp"]    = pd.to_datetime(rth_df["timestamp"], utc=True)
rth_df["ts_ny"]        = rth_df["timestamp"].dt.tz_convert("America/New_York")
rth_df["session_date"] = rth_df["ts_ny"].dt.date

print("Shape:", rth_df.shape)
print("Columns:")
print(rth_df.columns.tolist())

Shape: (171692, 30)
Columns:
['symbol', 'timestamp', 'date_ny', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'ret_1', 'ret_3', 'ret_6', 'ret_12', 'ret_18', 'ret_24', 'atr_12', 'atr_pct_12', 'vwap_session', 'vwap_dist', 'vwap_dist_atr', 'vol_ratio_18', 'range_pct', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'rv_12', 'tod_sin', 'tod_cos', 'ts_ny', 'session_date']


In [3]:
rth_df.head()

,symbol,timestamp,date_ny,open,high,low,close,volume,trade_count,ret_1,...,vol_ratio_18,range_pct,body_pct,upper_wick_pct,lower_wick_pct,rv_12,tod_sin,tod_cos,ts_ny,session_date
0,COIN,2024-02-28 19:00:00+00:00,2024-02-28,205.560,206.0450,204.94,205.4800,174393.0,2678.0,-0.000189,...,0.505886,0.005378,0.000389,0.002360,0.002628,0.007196,-0.935016,-0.354605,2024-02-28 14:00:00-05:00,2024-02-28
1,COIN,2024-02-28 19:05:00+00:00,2024-02-28,205.500,205.6337,204.66,204.8300,111220.0,1865.0,-0.003168,...,0.364770,0.004754,0.003271,0.000653,0.000830,0.006995,-0.960518,-0.278217,2024-02-28 14:05:00-05:00,2024-02-28
2,COIN,2024-02-28 19:10:00+00:00,2024-02-28,204.929,205.0621,203.52,203.7050,169909.0,2386.0,-0.005507,...,0.601525,0.007570,0.006009,0.000653,0.000908,0.007090,-0.979791,-0.200026,2024-02-28 14:10:00-05:00,2024-02-28
3,COIN,2024-02-28 19:15:00+00:00,2024-02-28,203.750,204.9900,203.52,204.4947,104130.0,1491.0,0.003869,...,0.395331,0.007188,0.003642,0.002422,0.001125,0.005988,-0.992709,-0.120537,2024-02-28 14:15:00-05:00,2024-02-28
4,COIN,2024-02-28 19:20:00+00:00,2024-02-28,204.550,204.7199,203.45,203.6800,95168.0,1327.0,-0.003992,...,0.387224,0.006235,0.004271,0.000834,0.001129,0.005532,-0.999189,-0.040266,2024-02-28 14:20:00-05:00,2024-02-28


In [4]:
# sort by stock and time stamp
rth_df = rth_df.sort_values(["ts_ny", "symbol"]).reset_index(drop=True)

print("Columns count:", len(rth_df.columns))
print("Symbols:", rth_df["symbol"].unique())
print("Date range:", rth_df["ts_ny"].min(), "to", rth_df["ts_ny"].max())

Columns count: 30
Symbols: <StringArray>
['TSLA', 'NVDA', 'PLTR', 'COIN', 'OKLO']
Length: 5, dtype: str
Date range: 2024-02-28 13:50:00-05:00 to 2026-02-27 11:55:00-05:00


In [5]:
# check for missing features
feature_cols = [
    "ret_1","ret_3","ret_6","ret_12","ret_18","ret_24",
    "atr_12","atr_pct_12",
    "vwap_session","vwap_dist","vwap_dist_atr",
    "vol_ratio_18",
    "range_pct","body_pct","upper_wick_pct","lower_wick_pct",
    "rv_12",
    "tod_sin","tod_cos"
]

missing_cols = [c for c in feature_cols if c not in rth_df.columns]
print("Missing feature columns:", missing_cols)

Missing feature columns: []


In [6]:
# validate price data is in chronological order
per_symbol_sorted = rth_df.groupby("symbol")["ts_ny"].apply(lambda s: s.is_monotonic_increasing)
print("\nEach Stock is in Chrono Order?")
print(per_symbol_sorted)


Each Stock is in Chrono Order?
symbol
COIN    True
NVDA    True
OKLO    True
PLTR    True
TSLA    True
Name: ts_ny, dtype: bool


In [7]:
# drop rows with NaNs features
mask_valid = rth_df[feature_cols].notna().all(axis=1)
print("Valid rows for clustering:", mask_valid.sum(), "/", len(rth_df))
print("Dropped rows due to NaNs:", (~mask_valid).sum())

Valid rows for clustering: 171692 / 171692
Dropped rows due to NaNs: 0


In [8]:
# verify price data range and daily trading sessions
print("Calendar date range:", rth_df["session_date"].min(), "->", rth_df["session_date"].max())
print("Unique trading sessions:", rth_df["session_date"].nunique())

Calendar date range: 2024-02-28 -> 2026-02-27
Unique trading sessions: 484


In [9]:
# calculate daily summary features

# ensure stocks are sorted by time stamp
rth_df = rth_df.sort_values(["symbol", "ts_ny"]).reset_index(drop=True)

# calculate return from close to close for each stock
rth_df["ret_cc"] = rth_df.groupby("symbol")["close"].pct_change()

# calculate daily price data for each stock
daily = (
    rth_df
    .groupby(["symbol", "session_date"])
    .agg(
        day_open=("open","first"),
        day_close=("close","last"),
        day_high=("high","max"),
        day_low=("low","min"),
        day_volume=("volume","sum"),
        bars_in_day=("close","size"),
        daily_rv=("ret_cc", lambda x: np.nanstd(x, ddof=0)),
    )
    .reset_index()
)

daily["daily_ret"] = daily["day_close"] / daily["day_open"] - 1.0
daily["daily_abs_ret"] = daily["daily_ret"].abs()
daily["daily_range_pct"] = (daily["day_high"] - daily["day_low"]) / daily["day_open"]
daily["daily_trend_strength"] = daily["daily_ret"].abs() / (daily["daily_range_pct"] + 1e-12)

print("Daily table shape:", daily.shape)
print("\nSample rows:")
print(daily.head())

Daily table shape: (2237, 13)

Sample rows:
  symbol session_date  day_open  day_close  day_high   day_low  day_volume  \
0   COIN   2024-02-28    205.56    200.730   206.045  199.2400   3674553.0   
1   COIN   2024-02-29    206.46    203.420   211.310  193.8800  14412653.0   
2   COIN   2024-03-01    202.70    205.830   206.390  196.0101   8514752.0   
3   COIN   2024-03-04    217.40    228.720   236.460  212.2469  20685818.0   
4   COIN   2024-03-05    230.00    216.774   239.980  215.4000  21519305.0   

   bars_in_day  daily_rv  daily_ret  daily_abs_ret  daily_range_pct  \
0           24  0.003538  -0.023497       0.023497         0.033105   
1           78  0.007627  -0.014724       0.014724         0.084423   
2           78  0.003980   0.015442       0.015442         0.051208   
3           78  0.008353   0.052070       0.052070         0.111376   
4           78  0.007343  -0.057504       0.057504         0.106870   

   daily_trend_strength  
0              0.709772  
1       

### Rule-Based Regime Classification

Define four trading regimes using threshold rules on the macro features computed above.

The 4 Regimes are Trending, Choppy, Range Bound, and Flat.
- Trending: Strong directional moves - wider targets, longer holds
    - conditions_3 > 0.60 and trend_feq >= 0.33
- Choppy: Volatile but non-directional - tight targets, quick exits
    - conditions_3 < 0.40 and rv_impluse > 0 
- Range Bound: Contained oscillation - moderate targets, conservative holds
    - conditions_3 < 0.40 and rv_impulse <= 0
- Flat: Minimal movement - smallest targets or skip
    - rv_impulse , -0.30 and range_ratios < -0.20 and rel_vol < -0.10

In [10]:
# Create rolling macro features
daily = daily.sort_values(["symbol","session_date"]).reset_index(drop=True)

# using 3 day rolling period
N = 3
LONG_N = 10
eps = 1e-8

# short term realized volatility
daily["rv_3"] = (
    daily.groupby("symbol")["daily_rv"]
    .transform(lambda s: s.shift(1).rolling(N).mean())
)

# long term realized volatility
daily["rv_long"] = (
    daily.groupby("symbol")["daily_rv"]
    .transform(lambda s: s.shift(1).rolling(LONG_N).mean())
)

# volatility expansion or contraction
daily["rv_impulse"] = np.log((daily["rv_3"] + eps) / (daily["rv_long"] + eps))

# range
daily["range_3"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .transform(lambda s: s.shift(1).rolling(N).mean())
)

# long term range
daily["range_long"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .transform(lambda s: s.shift(1).rolling(LONG_N).mean())
)

# range expansion or compression
daily["range_ratio"] = np.log((daily["range_3"] + eps) / (daily["range_long"] + eps))

# trend frequency
trend_thresh = 0.6
daily["trend_freq"] = (
    daily.groupby("symbol")["daily_trend_strength"]
    .transform(lambda s: (s.shift(1) > trend_thresh).rolling(N).mean())
)

# directional or chop
daily["sum_ret_3"] = (
    daily.groupby("symbol")["daily_ret"]
    .transform(lambda s: s.shift(1).rolling(N).sum())
)

daily["sum_abs_ret_3"] = (
    daily.groupby("symbol")["daily_abs_ret"]
    .transform(lambda s: s.shift(1).rolling(N).sum())
)

daily["conditions_3"] = daily["sum_ret_3"].abs() / (daily["sum_abs_ret_3"] + eps)

# relative volume vs past 10 days
daily["rel_vol"] = (
    daily.groupby("symbol")["day_volume"]
    .transform(lambda s: np.log(s.shift(1) / (s.shift(1).rolling(LONG_N).median() + eps)))
)

macro_feature_cols = [
    "rv_impulse",
    "range_ratio",
    "trend_freq",
    "conditions_3",
    "rel_vol"
]

print(daily[["symbol", "session_date"] + macro_feature_cols].head(15))

   symbol session_date  rv_impulse  range_ratio  trend_freq  conditions_3  \
0    COIN   2024-02-28         NaN          NaN         NaN           NaN   
1    COIN   2024-02-29         NaN          NaN         NaN           NaN   
2    COIN   2024-03-01         NaN          NaN    0.333333           NaN   
3    COIN   2024-03-04         NaN          NaN    0.333333      0.424497   
4    COIN   2024-03-05         NaN          NaN    0.000000      0.641898   
5    COIN   2024-03-06         NaN          NaN    0.000000      0.080047   
6    COIN   2024-03-07         NaN          NaN    0.000000      0.238468   
7    COIN   2024-03-08         NaN          NaN    0.000000      0.049591   
8    COIN   2024-03-11         NaN          NaN    0.000000      1.000000   
9    COIN   2024-03-12         NaN          NaN    0.333333      0.035393   
10   COIN   2024-03-13    0.050792     0.096355    0.333333      0.186174   
11   COIN   2024-03-14   -0.152740    -0.089752    0.333333      1.000000   

In [11]:
# normalize by stock looking back 20 days
norm_cols = macro_feature_cols.copy()
z_cols = []

Z_LOOKBACK = 20
eps = 1e-8

for col in norm_cols:
    zcol = f"{col}_z"
    daily[zcol] = (
        daily.groupby("symbol")[col]
        .transform(lambda s: (s - s.shift(1).rolling(Z_LOOKBACK).mean()) /
                             (s.shift(1).rolling(Z_LOOKBACK).std() + eps))
    )
    z_cols.append(zcol)

daily.tail(20)

,symbol,session_date,day_open,day_close,day_high,day_low,day_volume,bars_in_day,daily_rv,daily_ret,...,trend_freq,sum_ret_3,sum_abs_ret_3,conditions_3,rel_vol,rv_impulse_z,range_ratio_z,trend_freq_z,conditions_3_z,rel_vol_z
2217,TSLA,2026-01-22,435.190,449.3700,449.5000,432.6293,63597499.0,78,0.001825,0.032583,...,0.333333,-0.004924,0.051407,0.095786,0.112616,1.308573,1.282009,-0.255474,-2.401159,0.918018
2218,TSLA,2026-01-23,447.425,449.0700,452.4300,444.0400,51146282.0,78,0.001852,0.003677,...,0.666667,0.032278,0.079372,0.406673,0.176865,1.113234,1.380145,0.596107,-1.080229,1.128861
2219,TSLA,2026-01-26,445.020,435.2000,445.0500,434.2800,43782576.0,78,0.002705,-0.022066,...,0.333333,0.059502,0.059502,1.000000,-0.041020,0.371513,0.916864,-0.350115,0.732689,0.171761
2220,TSLA,2026-01-27,437.410,430.9100,437.5200,430.6900,33965748.0,78,0.001275,-0.014860,...,0.666667,0.014194,0.058326,0.243348,-0.184723,0.300192,0.186150,0.500436,-1.682424,-0.516053
2221,TSLA,2026-01-28,431.910,431.4600,438.2600,430.1000,42249391.0,78,0.001916,-0.001042,...,0.666667,-0.033250,0.040603,0.818901,-0.418231,-0.034513,-0.989999,0.428546,0.261950,-2.081487
2222,TSLA,2026-01-29,437.640,416.4400,440.2299,414.6200,70903529.0,78,0.002825,-0.048442,...,0.666667,-0.037969,0.037969,1.000000,-0.199994,-0.225924,-1.033791,0.382301,0.852936,-0.865661
2223,TSLA,2026-01-30,425.770,430.6200,439.8800,422.7000,74741961.0,78,0.003738,0.011391,...,0.666667,-0.064344,0.064344,1.000000,0.306076,-0.283485,0.433963,0.382301,0.852937,1.881676
2224,TSLA,2026-02-02,421.250,421.7850,427.1500,414.5000,51665167.0,78,0.004390,0.001270,...,0.333333,-0.038092,0.060875,0.625751,0.338332,1.306306,1.100547,-0.550725,-0.301790,1.807773
2225,TSLA,2026-02-03,424.270,421.8100,428.5600,413.6901,52032872.0,78,0.002471,-0.005798,...,0.333333,-0.035780,0.061103,0.585578,-0.015418,2.187361,1.417053,-0.476436,-0.375009,-0.160216
2226,TSLA,2026-02-04,420.400,405.8300,423.9000,399.1800,67848231.0,78,0.003329,-0.034657,...,0.000000,0.006863,0.018459,0.371788,0.003540,1.837975,0.297647,-1.542691,-1.003556,0.006454


In [12]:
# drop NaNs
macro_z_cols = ["rv_impulse","range_ratio","trend_freq","conditions_3","rel_vol",
                "rv_impulse_z","range_ratio_z","trend_freq_z","conditions_3_z","rel_vol_z"]
macro = daily.dropna(subset=macro_z_cols).copy()

print(macro.head())

   symbol session_date  day_open  day_close  day_high   day_low  day_volume  \
30   COIN   2024-04-11    250.99     263.07  264.3000  247.3103   7622950.0   
31   COIN   2024-04-12    259.42     245.79  259.9700  244.3101   7663576.0   
32   COIN   2024-04-15    247.94     223.36  249.7392  222.1400   9936693.0   
33   COIN   2024-04-16    223.00     218.84  223.0000  205.6700  14951181.0   
34   COIN   2024-04-17    221.93     213.54  224.8700  205.9000   8495762.0   

    bars_in_day  daily_rv  daily_ret  ...  trend_freq  sum_ret_3  \
30           78  0.004016   0.048129  ...    0.666667   0.031839   
31           78  0.004723  -0.052540  ...    1.000000   0.062226   
32           78  0.004553  -0.099137  ...    1.000000   0.049358   
33           78  0.006594  -0.018655  ...    1.000000  -0.103548   
34           78  0.005566  -0.037805  ...    0.666667  -0.170332   

    sum_abs_ret_3  conditions_3   rel_vol  rv_impulse_z  range_ratio_z  \
30       0.111185      0.286362 -0.028097 

In [13]:
entries = pd.read_csv("multiasset_labeled_entries.csv")
entries["ts_entry"] = pd.to_datetime(entries["ts_entry"], utc=True)

min_val_ts  = entries.loc[entries["split"]=="val", "ts_entry"].min()
min_test_ts = entries.loc[entries["split"]=="test", "ts_entry"].min()

first_val_day  = min_val_ts.tz_convert("America/New_York").date()
first_test_day = min_test_ts.tz_convert("America/New_York").date()

print("first_val_day :", first_val_day)
print("first_test_day:", first_test_day)

first_val_day : 2025-07-15
first_test_day: 2025-10-28


In [14]:
# split macro data
macro_train = macro[macro["session_date"] < first_val_day].copy()
macro_val   = macro[(macro["session_date"] >= first_val_day) & (macro["session_date"] < first_test_day)].copy()
macro_test  = macro[macro["session_date"] >= first_test_day].copy()

print("Macro train rows:", len(macro_train))
print("Macro val rows  :", len(macro_val))
print("Macro test rows :", len(macro_test))

Macro train rows: 1399
Macro val rows  : 335
Macro test rows : 353


#### Threshold Definitions and Regime Assignment

In [15]:
# regime thresholds
T = {
    'flat_rv'     : -0.30,   # rv_impulse: very compressed short-term vol vs long-run
    'flat_range'  : -0.20,   # range_ratio: very narrow range vs history
    'flat_vol'    : -0.10,   # rel_vol: below-average volume
    'trend_cond3' :  0.60,   # conditions_3: strong directional consistency
    'trend_freq'  :  0.33,   # trend_freq: >= 1 of last 3 days was trending
    'chop_cond3'  :  0.40,   # conditions_3: weak directional consistency
    'chop_rv'     :  0.00,   # rv_impulse: short-term vol expanding vs long-run avg
}

# regime labels
def label_regimes(df, t):
    regime = pd.Series('Range Bound', index=df.index, dtype='object')

    choppy   = (df['conditions_3'] < t['chop_cond3'])  & (df['rv_impulse'] > t['chop_rv'])
    trending = (df['conditions_3'] > t['trend_cond3']) & (df['trend_freq'] >= t['trend_freq'])
    flat     = (df['rv_impulse']   < t['flat_rv'])      & (df['range_ratio'] < t['flat_range']) \
             & (df['rel_vol']      < t['flat_vol'])

    regime[choppy]   = 'Choppy'
    regime[trending] = 'Trending'
    regime[flat]     = 'Flat'

    return regime

# apply regime labels
macro['regime_label'] = label_regimes(macro, T)

# encode regimes
REGIME_INT = {'Flat': 0, 'Trending': 1, 'Choppy': 2, 'Range Bound': 3}
macro['macro_regime_id'] = macro['regime_label'].map(REGIME_INT)

print("Regime distribution:")
dist = macro['regime_label'].value_counts()
frac = macro['regime_label'].value_counts(normalize=True).round(3)
print(pd.DataFrame({'count': dist, 'fraction': frac}))

Regime distribution:
              count  fraction
regime_label                 
Range Bound     891     0.427
Trending        726     0.348
Choppy          339     0.162
Flat            131     0.063


In [16]:
# check regime distribution
macro_train_mask = macro['session_date'] < first_val_day
val_mask  = (macro['session_date'] >= first_val_day) & (macro['session_date'] < first_test_day)
test_mask = macro['session_date'] >= first_test_day

for name, mask in [('Train', macro_train_mask), ('Val', val_mask), ('Test', test_mask)]:
    dist = macro.loc[mask, 'regime_label'].value_counts()
    frac = (dist / dist.sum()).round(3)
    print(f"\n{name} set ({mask.sum()} rows):")
    print(pd.DataFrame({'count': dist, 'fraction': frac}))


Train set (1399 rows):
              count  fraction
regime_label                 
Range Bound     587     0.420
Trending        468     0.335
Choppy          240     0.172
Flat            104     0.074

Val set (335 rows):
              count  fraction
regime_label                 
Range Bound     156     0.466
Trending        119     0.355
Choppy           55     0.164
Flat              5     0.015

Test set (353 rows):
              count  fraction
regime_label                 
Range Bound     148     0.419
Trending        139     0.394
Choppy           44     0.125
Flat             22     0.062


In [17]:
# validate regimes
print("Mean macro features per regime:")
display(
    macro.loc[macro_train_mask]
    .groupby('regime_label')[macro_feature_cols]
    .mean()
    .round(4)
    .sort_values('trend_freq', ascending=False)
)

Mean macro features per regime:


,rv_impulse,range_ratio,trend_freq,conditions_3,rel_vol
regime_label,,,,,
Trending,-0.0100,0.0313,0.4373,0.8845,0.1254
Choppy,0.2005,0.0814,0.3694,0.2029,0.1293
Range Bound,-0.1041,-0.0556,0.2618,0.4850,-0.0062
Flat,-0.4770,-0.4475,0.2212,0.5654,-0.3978


In [18]:
# Regime persistence and transitions
macro_sorted = macro.sort_values(['symbol', 'session_date']).copy()
macro_sorted['prev_regime'] = macro_sorted.groupby('symbol')['regime_label'].shift(1)

valid = macro_sorted.dropna(subset=['prev_regime'])
persistence = (valid['regime_label'] == valid['prev_regime']).mean()
print(f"Day-over-day regime persistence: {persistence:.1%}\n")

print("Transition matrix (row = yesterday > col = today):")
trans = pd.crosstab(
    valid['prev_regime'],
    valid['regime_label'],
    normalize='index'
).round(3)
display(trans)

Day-over-day regime persistence: 51.4%

Transition matrix (row = yesterday > col = today):


regime_label,Choppy,Flat,Range Bound,Trending
prev_regime,,,,
Choppy,0.345,0.006,0.413,0.236
Flat,0.015,0.511,0.282,0.191
Range Bound,0.137,0.048,0.557,0.257
Trending,0.133,0.025,0.301,0.541


In [19]:
# save regime thresholds
regime_meta = {
    "thresholds": T,
    "regime_int_map": REGIME_INT,
}
with open("regime_thresholds.json", "w") as f:
    json.dump(regime_meta, f, indent=2)

print("Saved regime_thresholds.json")
print("\nInteger encoding used:")
for label, code in REGIME_INT.items():
    count = (macro['regime_label'] == label).sum()
    print(f"  {code} = {label:<12}  ({count} total session-days)")

Saved regime_thresholds.json

Integer encoding used:
  0 = Flat          (131 total session-days)
  1 = Trending      (726 total session-days)
  2 = Choppy        (339 total session-days)
  3 = Range Bound   (891 total session-days)


In [20]:
# add regime to 5M price data
rth_df = rth_df.merge(
    macro[["symbol", "session_date", "macro_regime_id", "regime_label"]],
    on=["symbol", "session_date"],
    how="left"
)

print("Missing macro_regime_id on bars:", rth_df["macro_regime_id"].isna().sum())
print("Bars shape:", rth_df.shape)
print(rth_df[["symbol", "ts_ny", "session_date", "macro_regime_id", "regime_label"]].head(10))

Missing macro_regime_id on bars: 10762
Bars shape: (171692, 33)
  symbol                     ts_ny session_date  macro_regime_id regime_label
0   COIN 2024-02-28 14:00:00-05:00   2024-02-28              NaN          NaN
1   COIN 2024-02-28 14:05:00-05:00   2024-02-28              NaN          NaN
2   COIN 2024-02-28 14:10:00-05:00   2024-02-28              NaN          NaN
3   COIN 2024-02-28 14:15:00-05:00   2024-02-28              NaN          NaN
4   COIN 2024-02-28 14:20:00-05:00   2024-02-28              NaN          NaN
5   COIN 2024-02-28 14:25:00-05:00   2024-02-28              NaN          NaN
6   COIN 2024-02-28 14:30:00-05:00   2024-02-28              NaN          NaN
7   COIN 2024-02-28 14:35:00-05:00   2024-02-28              NaN          NaN
8   COIN 2024-02-28 14:40:00-05:00   2024-02-28              NaN          NaN
9   COIN 2024-02-28 14:45:00-05:00   2024-02-28              NaN          NaN


In [21]:
# create regime dataset
rth_regime = rth_df.dropna(subset=["macro_regime_id"]).copy()
rth_regime["macro_regime_id"] = rth_regime["macro_regime_id"].astype(int)

print("Regime bars:", rth_regime.shape)
print("Dropped bars:", len(rth_df) - len(rth_regime))

Regime bars: (160930, 33)
Dropped bars: 10762


In [22]:
# add regimes to VWAP Reclaim Long labeled entries csv
entries_df = pd.read_csv("multiasset_labeled_entries.csv")
entries_df["ts_entry"]     = pd.to_datetime(entries_df["ts_entry"], utc=True)
entries_df["session_date"] = entries_df["ts_entry"].dt.tz_convert("America/New_York").dt.date

entries_df = entries_df.merge(
    macro[["symbol", "session_date", "macro_regime_id", "regime_label"]],
    on=["symbol", "session_date"],
    how="left"
)

print("Entries:", len(entries_df))
print("Entries missing macro_regime_id:", entries_df["macro_regime_id"].isna().sum())

Entries: 1616
Entries missing macro_regime_id: 108


In [23]:
# create entries with regime dataset
entries_regime = entries_df.dropna(subset=["macro_regime_id"]).copy()
entries_regime["macro_regime_id"] = entries_regime["macro_regime_id"].astype(int)

print("Regime entries:", len(entries_regime))

Regime entries: 1508


In [24]:
# save new datasets
rth_regime.drop(columns=["ts_ny"], errors="ignore").to_csv(
    "multiasset_5m_rth_features_with_regime.csv", index=False
)
entries_regime.to_csv("multiasset_labeled_entries_with_regime.csv", index=False)

In [25]:
# save daily macro dataset
macro.to_csv("macro_daily_with_regimes.csv", index=False)
print(macro.shape)

(2087, 31)
